# 第9天：第一次接触机器学习

今天不背算法公式。我们先用6名用户理解人工规则如何判断，再把真实数据整理成第10天可以训练的形式。

> 机器学习：让计算机从许多带答案的历史样本中学习规律，再判断没有见过的新样本。

## 任务0：环境和个人信息

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.20
pd.set_option("display.max_columns", 80)

In [ ]:
# TODO 9-0：填写个人信息
STUDENT_NAME = "谢xx"
STUDENT_ID = "24040092"
CLASS_NAME = "电商数据分析"
assert STUDENT_NAME and STUDENT_ID and CLASS_NAME, "请先填写个人信息"

## 任务1：先用6名用户理解“规则”和“预测”

人工规则：使用月数不超过3个月，并且发生过投诉，就预测为流失。`Tenure`和`Complain`是判断依据，`Churn`是真实答案。

In [ ]:
toy = pd.DataFrame({
    "用户": list("ABCDEF"),
    "Tenure": [1, 24, 3, 18, 2, 30],
    "Complain": [1, 0, 0, 1, 1, 0],
    "Churn": [1, 0, 1, 0, 1, 0],
})
toy["人工规则预测"] = ((toy["Tenure"] <= 3) & (toy["Complain"] == 1)).astype(int)
toy["是否判断正确"] = toy["人工规则预测"] == toy["Churn"]
display(toy)
print("判断正确：", int(toy["是否判断正确"].sum()), "/", len(toy))
print("真实流失3人，规则找到：", int(((toy["人工规则预测"] == 1) & (toy["Churn"] == 1)).sum()), "人")

**思考：** C用户为什么被漏掉？人工规则与机器学习有什么不同？

C用户被漏掉是因为他使用月数≤3但没有投诉，不满足人工规则的两个条件。人工规则依赖人类经验设定，只能捕捉简单的线性关系，容易遗漏复杂模式。机器学习则从数据中自动学习特征与标签之间的关联，能够发现人类难以察觉的非线性关系。

## 任务2：读取真实数据并认识样本与标签

In [ ]:
df = pd.read_csv(DATA_PATH)
print("数据形状：", df.shape)
print("一行代表一名用户；总体流失率：", f"{df['Churn'].mean():.2%}")
assert df.shape == (5630, 22)
assert df["CustomerID"].is_unique
assert set(df["Churn"].unique()) == {0, 1}
assert int(df.isna().sum().sum()) == 0

## 任务3：建立X和y

`X`是模型可以查看的特征，`y`是希望预测的答案。

In [ ]:
TARGET = "Churn"
# TODO 9-1：用户编号字段是什么？
ID_COL = "CustomerID"
assert ID_COL == "CustomerID", "请填写正确的用户编号字段"
X = df.drop(columns=[TARGET, ID_COL]).copy()
y = df[TARGET].astype(int).copy()
assert TARGET not in X.columns and ID_COL not in X.columns
print("特征表：", X.shape, "标签：", y.shape)

## 任务4：查看特征方案

代码自动区分数值列、文字类别和派生特征。今天重点理解处理目的，不要求默写。

In [ ]:
categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(exclude="object").columns.tolist()
derived_features = ["TenureGroup", "IsMobileLogin"]

rows = []
for column in df.columns:
    if column == ID_COL:
        role, action, reason = "identifier", "drop", "用户编号只用于追踪"
    elif column == TARGET:
        role, action, reason = "target", "separate", "这是希望预测的答案"
    elif column in derived_features:
        role, action, reason = "derived_feature", "candidate", "由已有字段转换得到"
    elif column in categorical_features:
        role, action, reason = "categorical_feature", "one_hot", "文字类别需要转成0/1列"
    else:
        role, action, reason = "numeric_feature", "numeric_pipeline", "进入教师提供的数值处理分支"
    rows.append({"feature": column, "role": role, "dtype": str(df[column].dtype), "action": action, "reason": reason})

feature_schema = pd.DataFrame(rows)
feature_schema.to_csv(OUTPUT_DIR / "feature_schema.csv", index=False, encoding="utf-8-sig")
display(feature_schema)

## 任务5：把数据分成训练集和测试集

训练集用于学习，测试集模拟没有见过的新用户。

In [ ]:
# TODO 9-2：为了让两部分流失比例接近，把None改为y
STRATIFY_TARGET = y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=STRATIFY_TARGET)
assert STRATIFY_TARGET is y, "请使用y进行分层划分"

In [ ]:
split_summary = pd.DataFrame([
    {"split": "train", "rows": len(X_train), "churn_count": int(y_train.sum()), "churn_rate": y_train.mean()},
    {"split": "test", "rows": len(X_test), "churn_count": int(y_test.sum()), "churn_rate": y_test.mean()},
])
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False, encoding="utf-8-sig")
display(split_summary)
assert len(X_train) + len(X_test) == 5630
assert abs(y_train.mean() - y_test.mean()) < 0.01

## 任务6：运行教师提供的预处理流水线

- 数值分支：缺失兜底并统一尺度；
- 类别分支：把文字类别转为0/1列；
- 流水线只能从训练集学习规则。

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

# fit_transform：在训练集学习规则并完成加工
X_train_ready = preprocessor.fit_transform(X_train)
# transform：测试集只使用已经学到的规则
X_test_ready = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

assert X_train_ready.shape[1] == X_test_ready.shape[1] == 36
assert np.isfinite(X_train_ready).all() and np.isfinite(X_test_ready).all()
model_matrix_preview = pd.DataFrame(X_train_ready[:20], columns=feature_names)
model_matrix_preview.to_csv(OUTPUT_DIR / "model_matrix_preview.csv", index=False, encoding="utf-8-sig")
print("训练矩阵：", X_train_ready.shape, "测试矩阵：", X_test_ready.shape)
display(model_matrix_preview.head())

## 任务7：运行最低参照线

它永远预测人数最多的类别，只用来回答：正式模型至少要超过什么水平？

In [ ]:
baseline = DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
baseline.fit(X_train_ready, y_train)
y_pred = baseline.predict(X_test_ready)

baseline_metrics = pd.DataFrame({
    "metric": ["accuracy", "churn_recall", "predicted_churn_count"],
    "value": [accuracy_score(y_test, y_pred), recall_score(y_test, y_pred, pos_label=1, zero_division=0), int(y_pred.sum())],
})
baseline_metrics.to_csv(OUTPUT_DIR / "baseline_metrics.csv", index=False, encoding="utf-8-sig")
display(baseline_metrics)
print("测试集中真实流失人数：", int(y_test.sum()))
print("最低参照线预测流失人数：", int(y_pred.sum()))

## 任务8：写出自己的解释

In [ ]:
# TODO 9-3：用100～200字解释特征、标签、训练集、测试集和最低参照线
reflection = "特征是用来预测的输入信息，如用户的使用月数和投诉记录；标签是要预测的目标答案，如用户是否流失。训练集是用来让模型学习规律的历史数据，测试集是用来评估模型效果的未知数据，两者必须分开才能避免过拟合。最低参照线是一个简单的基准模型，它永远预测最常见的类别，用来衡量正式模型是否有价值。如果正式模型的性能低于或接近最低参照线，说明模型没有学到有用的规律。"
assert 100 <= len(reflection) <= 200, "请完成100～200字复盘"
print("解释字数：", len(reflection))
print(reflection)

## 提交检查

In [ ]:
required = {"feature_schema.csv", "split_summary.csv", "model_matrix_preview.csv", "baseline_metrics.csv"}
actual = {path.name for path in OUTPUT_DIR.glob("*.csv")}
print("成果文件：", sorted(actual))
assert required.issubset(actual)